# 提交说明（Chapter 4）

本 notebook 对应“主数据富化与类别对齐（E-commerce）”机制实现。

- 使用参数单元管理输入/输出路径。
- 提交前请确认不包含内部路径和敏感数据。

In [ ]:
# 参数区（提交版本）
import os
from pathlib import Path

DATA_ROOT = Path(os.environ.get("CH5_DATA_ROOT", "./data_sample"))
OUTPUT_ROOT = Path(os.environ.get("CH5_OUTPUT_ROOT", "./outputs"))

for p in [DATA_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

In [ ]:
# Global Cateogry Mapping for SmartPath
# Eric Wu, GM Office, BCH China
# Jul,2024

In [ ]:
import pandas as pd
import difflib

In [ ]:
path_input = '//bcnshgs0295/02_FlatDataSource/01_UploadFile/82_GM_Office/02_Data_Platform/06_SmartPath/'
path_output = '//bcnshgs0295/02_FlatDataSource/01_UploadFile/82_GM_Office/02_Data_Platform/06_SmartPath/'

In [ ]:
df_mapping_list = pd.read_excel(path_input + "SKU_Global_Mapping_List.xlsx")
df_global_mapping = pd.read_excel(path_input + "Global_Mapping_Latest.xlsx")
#df_global_mapping_reference = pd.read_excel(path_input + "Global_Mapping_Historical.xlsx")

In [ ]:
df_mapping_list = df_mapping_list.drop_duplicates(subset=['SKU ID'], keep='first')

In [ ]:
merged_df = pd.merge(df_global_mapping, df_mapping_list, on='SKU ID', how='left')

df_global_mapping_reference = merged_df.copy()
df_global_mapping_reference = df_global_mapping_reference[df_global_mapping_reference['Global Category'].notna()]


merged_df['New SKU'] = False
merged_df['Comments'] = ""

for index, row in merged_df.iterrows():
    
    if pd.isnull(row['Global Category']):

        conditions = (
            (df_global_mapping_reference['Bayer Category III'] == row['Bayer Category III']) &
            (df_global_mapping_reference['Formats'] == row['Formats']) &
            (df_global_mapping_reference['Bayer_Consumer Type'] == row['Bayer_Consumer Type']) &
            (df_global_mapping_reference['Subcategory'] == row['Subcategory']) &
            (df_global_mapping_reference['Bayer_Report Category'] == row['Bayer_Report Category']) &
            (df_global_mapping_reference['Bayer_Product Usage'] == row['Bayer_Product Usage']) &
            (df_global_mapping_reference['Bayer_HF/OTC'] == row['Bayer_HF/OTC']) &
            (df_global_mapping_reference['SKU ID'] != row['SKU ID'])
            
        )
    
        if not(pd.isnull(row['Product attributes'])):
            conditions =  conditions & (df_global_mapping_reference['Product attributes'] == row['Product attributes'])
            
        filtered_df = df_global_mapping_reference[conditions]
        unique_combinations = filtered_df.drop_duplicates(subset=['Global Category', 'Global subCategory', 'Global segment'])
     
        if len(unique_combinations) == 1:

            merged_df.at[index, 'Global Category'] = unique_combinations.iloc[0]['Global Category']
            merged_df.at[index, 'Global subCategory'] = unique_combinations.iloc[0]['Global subCategory']
            merged_df.at[index, 'Global segment'] = unique_combinations.iloc[0]['Global segment']
            merged_df.at[index, 'New SKU'] = True
            merged_df.at[index, 'Comments'] = "Conditional Match"
            
        else:

            merged_df.at[index, 'Global Category'] = 'to be mapped'
            merged_df.at[index, 'Global subCategory'] = 'to be mapped'
            merged_df.at[index, 'Global segment'] = 'to be mapped'
            merged_df.at[index, 'New SKU'] = True

In [ ]:
to_be_mapped_rows = merged_df['Global Category'] == 'to be mapped'

for index, row in merged_df[to_be_mapped_rows].iterrows():
    same_category_rows = (
    
    (df_global_mapping_reference['Bayer Category III'] == row['Bayer Category III']) &
    (df_global_mapping_reference['Bayer_Consumer Type'] == row['Bayer_Consumer Type']) &
    (df_global_mapping_reference['Subcategory'] == row['Subcategory']) &
    #(df_global_mapping_reference['Bayer_Report Category'] == row['Bayer_Report Category']) &
    (df_global_mapping_reference['Bayer_Product Usage'] == row['Bayer_Product Usage']) &
    #(df_global_mapping_reference['Bayer_HF/OTC'] == row['Bayer_HF/OTC']) &
    (df_global_mapping_reference['SKU ID'] != row['SKU ID'])
            
    )
    
    if same_category_rows.any():
        candidate_names = df_global_mapping_reference[same_category_rows]['SKU Name CN']
        closest_match = difflib.get_close_matches(row['SKU Name CN'], candidate_names, n=1, cutoff=0.1)
        
        if closest_match:
            match_index = df_global_mapping_reference[same_category_rows & (df_global_mapping_reference['SKU Name CN'] == closest_match[0])].index[0]
            
            merged_df.at[index, 'Global Category'] = df_global_mapping_reference.at[match_index, 'Global Category']
            merged_df.at[index, 'Global subCategory'] = df_global_mapping_reference.at[match_index, 'Global subCategory']
            merged_df.at[index, 'Global segment'] = df_global_mapping_reference.at[match_index, 'Global segment']
            merged_df.at[index, 'Comments'] = "AI Match"

In [ ]:
to_be_mapped_count = (merged_df['Global Category'] == 'to be mapped').sum()
to_be_mapped_count

In [ ]:
merged_df.loc[(merged_df['New SKU'] == True) & (merged_df['Comments'] == ""), 'Comments'] = "to be mapped manually"

In [ ]:
merged_df.to_excel(path_output + 'Global_Category_Mapping_List.xlsx',index=False)

### End